# Poralekha Bondhu — Run Muse Glimmer on Kaggle (free GPU)

This notebook runs Meta's open-weight model **Muse Glimmer** on Kaggle's free GPUs and gives the Poralekha Bondhu app a live API link.

**Before running anything, check the right-hand Settings pane:**
- **Accelerator: GPU T4 x2**
- **Internet: ON**

Then run the cells **from top to bottom, one at a time**. Wait for each cell to finish before running the next.

## Step 1 — Check the GPUs

In [ ]:
!nvidia-smi

## Step 2 — Install Ollama (the program that runs the model)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## Step 3 — Start the Ollama server

The model files are stored in `/kaggle/working`, Kaggle's saved space, so the big download does not have to happen again in later sessions **if you use File → Save Version at the end**.

In [ ]:
import os, subprocess, time

os.environ["OLLAMA_MODELS"] = "/kaggle/working/ollama_models"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.makedirs("/kaggle/working/ollama_models", exist_ok=True)

log = open("/kaggle/working/ollama.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ))
time.sleep(8)
print("Ollama server started.")

## Step 4 — Download Muse Glimmer (about 18 GB — this is the long step, wait for it to finish)

In [ ]:
!ollama pull muse-glimmer:30b-q4_K_M

## Step 5 — Test the model right here (it should answer in Bangla)

In [ ]:
import requests

r = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": "muse-glimmer:30b-q4_K_M",
        "messages": [{"role": "user", "content": "বাংলায় এক লাইনে উত্তর দাও: তুমি কে?"}],
    },
    timeout=900,
)
print("Status:", r.status_code)
print(r.json()["choices"][0]["message"]["content"])

## Step 6 — Make the public link for the app

This creates a temporary public address for the model. Run the cell; it prints exactly what to paste into the app's settings (gear icon). **Do not post this link publicly** — it works only while this session is running, and a fresh one is made each session.

In [ ]:
import subprocess, re, time, os

if not os.path.exists("/tmp/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared",
        shell=True,
    )

tlog = open("/kaggle/working/tunnel.log", "w")
subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=tlog,
    stderr=subprocess.STDOUT,
)
time.sleep(20)

text = open("/kaggle/working/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
if urls:
    print("=== PASTE THIS IN THE APP (gear icon / settings) ===")
    print("API URL:", urls[-1] + "/v1/chat/completions")
    print("Model:   muse-glimmer:30b-q4_K_M")
    print("API key: ollama")
else:
    print("Tunnel not ready yet - wait 20 seconds and run this cell again.")

## If something goes wrong

Run the cell below and share a screenshot of the output — it shows the server's error log.

In [ ]:
!tail -30 /kaggle/working/ollama.log; echo ---; tail -5 /kaggle/working/tunnel.log

## At the end

Use **File → Save Version → Save** so the downloaded model is kept for the next session (demo-video days) and does not need downloading again.

Model: Muse Glimmer (Meta open-weight, Apache-2.0), Q4_K_M build, served by Ollama on Kaggle's free T4 x2 GPUs, operated by MD Rifat Hasan.